# Building dust maps from the stellar catalog

The Zhang & Green (2025) dust maps in `Rv_map_new.h5` have
fixed resolutions (`nside` 64, 128, 256 and combined) and a fixed set of 25 distance bins. This
notebook builds equivalent maps directly from the stellar catalog so HEALPix resolution and distance binning can be customized. The new map is written to an HDF5 file.

Contents:

1. Load the stellar catalog
2. Choose a resolution and a distance binning
3. Define stellar catalog to HEALPix function
4. Extinction maps
5. Star count maps
6. Write the maps to HDF5
7. Read the maps back
8. Plot

In [ ]:
# Autoreload so we don't have to restart the kernel every time we change source code
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
# Astronomy packages
import h5py as h5
import healpy as hp
import astropy.units as u
from astropy.coordinates import SkyCoord
# My packages
# Add src/my_package to the import path (the repo root is one level up)
import sys
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src', 'my_package'))
if _src not in sys.path:
    sys.path.insert(0, _src)
import loader
from constants import data_dir, plt_dir

## 1. Load the stellar catalog

In [ ]:
"""Initialize a custom loader object for the stellar catalog.
    Rerunning this cell will initialize a new object, removing cached data from memory."""
catalog = loader.StellarCatalogLoader(data_dir + '/xpparams_v2_zenodo')

In [ ]:
"""Load the required columns from the Zhang and Green 2025 stellar catalog.
    Requested columns are cached in RAM; only columns not already in cache are read from disk.
"""
ra_all, dec_all, parallax_all, ext_all, quality_flags_all = catalog.get_columns(
    ['ra', 'dec', 'parallax', 'ext', 'quality_flags'])
print(f"{ra_all.size:,} stars in the catalog")

In [ ]:
"""Initialize objects that represent the CMB and Galactic BK fields."""
cmb = loader.Field('CMB', data_dir)
gal = loader.Field('GAL', data_dir)

Zhang & Green recommend a basic reliability cut of `quality_flags < 8`, which is used below.

In [ ]:
QUALITY_MAX = 8   # Basic reliability cut recommended by Zhang & Green 2025
qmask = quality_flags_all < QUALITY_MAX
print(f"{qmask.sum():,} of {qmask.size:,} stars pass quality_flags < {QUALITY_MAX} "
      f"({100 * qmask.mean():.1f}%)")

## 2. Choose a resolution and a distance binning

In [ ]:
"""The 25 distance bins (26 edges, in kpc) used by Zhang and Green 2025.
    Any monotonically increasing array of edges can be used in place of these."""
ZhangGreen = loader.HDF5Loader(data_dir, 'Rv_map_new.h5')
all_distances = ZhangGreen.load_data(['distance_bins'])

In [ ]:
# Resolution and binning used for the rest of this notebook.
NSIDE = 128
NEST = True    # HEALPix ordering
COORD = 'G'    # 'G' = galactic, 'C' = celestial/ICRS

dist_edges = all_distances                                # reproduce Zhang & Green
# dist_edges = np.array([0.0, 0.2, 0.5, 1.0, 2.0, 5.0])   # ...or supply your own

print(f"nside = {NSIDE}, npix = {hp.nside2npix(NSIDE):,}, "
      f"n_bins = {len(dist_edges) - 1}")

## 3. Define catalog to HEALPix function

`get_distance_mask` converts parallax to distance (`d = 1 / parallax`) and returns a
boolean mask for the stars inside a distance shell.

`hp_from_stars` finds the HEALPix pixel each star falls into and then either counts
stars per pixel (`stellar_param=None`) or averages a stellar parameter per pixel.
In the averaging case, pixels containing no stars are set to `NaN`.

In [ ]:
def get_distance_mask(dist_lims, parallax):
    """Boolean mask selecting stars whose distance falls inside `dist_lims` (kpc)."""
    star_dist = 1 / parallax
    return (dist_lims[0] < star_dist) & (star_dist < dist_lims[1])

In [ ]:
def hp_from_stars(nside, nest, coord, ra, dec, stellar_param=None):
    """Bin stellar catalog into a HEALPix map.
    If stellar_param is None, each pixel takes on the star count.
    Otherwise, each pixel takes on the mean of stellar_param for stars in that pixel.

    coord : 'C' for celestial/ICRS, 'G' for galactic
    """
    npix = hp.nside2npix(nside)
    if coord == 'C':
        lon, lat = ra, dec
    elif coord == 'G':
        coords = SkyCoord(ra=ra * u.degree, dec=dec * u.degree, frame='icrs')
        coords = coords.transform_to('galactic')
        lon, lat = coords.l.value, coords.b.value
    else:
        raise ValueError(f"Coordinate system {coord} not supported.")

    hp_indices = hp.ang2pix(nside, lon, lat, lonlat=True, nest=nest)

    if stellar_param is None:
        # count stars per pixel
        hp_map = np.bincount(hp_indices, minlength=npix).astype(float)
    else:
        # mean of a stellar parameter per pixel.
        # Flatten in case the parameter arrives as a column vector.
        param_flat = np.asarray(stellar_param).ravel()
        valid = ~np.isnan(param_flat)
        if not np.all(valid):
            hp_indices = hp_indices[valid]
            param_flat = param_flat[valid]
        counts = np.bincount(hp_indices, minlength=npix)
        sums = np.bincount(hp_indices, weights=param_flat, minlength=npix)
        # Vectorized mean; pixels with no stars become NaN
        hp_map = np.divide(sums, counts, out=np.full(npix, np.nan), where=counts > 0)
    return hp_map

## 4. Create extinction maps

The catalog's `ext` column is the extinction due to dust from the observer to
each star. Thus, averaging `ext` over the stars inside distance shell *i* gives
the integrated extinction. Turning integrated maps into 
differential maps is the subject of `differential_maps.ipynb`.

In [ ]:
def E_map_from_stars(nside, dist_edges, coord='G', nest=True, quality_max=8, verbose=True):
    """Build integrated extinction map from stellar catalog."""
    npix = hp.nside2npix(nside)
    n_bins = len(dist_edges) - 1
    E_map = np.empty((n_bins, npix))
    qmask = quality_flags_all < quality_max
    for i in range(n_bins):
        mask = get_distance_mask(dist_edges[i:i + 2], parallax_all) & qmask
        E_map[i] = hp_from_stars(nside, nest, coord,
                                 ra_all[mask], dec_all[mask], ext_all[mask])
        if verbose:
            print(f"bin {i:2d}  {dist_edges[i]:6.3f} - {dist_edges[i + 1]:6.3f} kpc  "
                  f"{mask.sum():>9,} stars")
    return E_map

In [ ]:
E_map_int = E_map_from_stars(NSIDE, dist_edges, coord=COORD, nest=NEST,
                             quality_max=QUALITY_MAX)
print(E_map_int.shape)

## 5. Star count maps

Call `hp_from_stars` with no `stellar_param` to produce star count maps.

- the integrated map: each pixel includes every star in that pixel regardless of distance. Shape  `(npix,)`.
- differential map: separated into distance slices. Shape `(n_bins, npix)`.

In [ ]:
def star_counts_from_stars(nside, dist_edges=None, coord='G', nest=True, quality_max=8, verbose = True):
    """Star counts per HEALPix pixel.

    If `dist_edges` is None, returns one integrated map of shape (npix,).
    Otherwise returns one map per distance shell, shape (len(dist_edges) - 1, npix)."""
    qmask = quality_flags_all < quality_max
    if dist_edges is None:
        return hp_from_stars(nside, nest, coord, ra_all[qmask], dec_all[qmask])

    npix = hp.nside2npix(nside)
    n_bins = len(dist_edges) - 1
    counts = np.empty((n_bins, npix))
    for i in range(n_bins):
        mask = get_distance_mask(dist_edges[i:i + 2], parallax_all) & qmask
        counts[i] = hp_from_stars(nside, nest, coord, ra_all[mask], dec_all[mask])
        if verbose:
                    print(f"bin {i:2d}  {dist_edges[i]:6.3f} - {dist_edges[i + 1]:6.3f} kpc  "
                          f"{mask.sum():>9,} stars")
    return counts

In [ ]:
star_counts_int = star_counts_from_stars(NSIDE, coord=COORD, nest=NEST,
                                         quality_max=QUALITY_MAX)
star_counts_diff = star_counts_from_stars(NSIDE, dist_edges, coord=COORD, nest=NEST,
                                          quality_max=QUALITY_MAX)
print(star_counts_int.shape, star_counts_diff.shape)
print(f"median stars per pixel (all distances): {np.median(star_counts_int):.0f}")

## 6. Write the maps to HDF5

`overwrite=True` deletes the old dataset before writing a new one. HDF5 does not reclaim the space on
disk, so repeatedly overwriting datasets makes the file grow. Create a new file if space is an issue.

In [ ]:
OUT_FILE = 'my_maps_tutorial.h5'


def write_map(fname, dataset_path, data, overwrite=False):
    """Write a HEALPix map to `data_dir/fname` at `dataset_path`, creating groups as needed."""
    path = os.path.join(data_dir, fname)
    with h5.File(path, 'a') as f:
        if dataset_path in f:
            if not overwrite:
                raise KeyError(f"{dataset_path} already exists in {fname}; "
                               f"pass overwrite=True to replace it.")
            del f[dataset_path]
        f.create_dataset(dataset_path, data=data)
    print(f"wrote {dataset_path} {data.shape} to {fname}")

In [ ]:
write_map(OUT_FILE, f'E_map/E_map_int_{NSIDE}', E_map_int)
write_map(OUT_FILE, f'star_counts/star_counts_int_{NSIDE}', star_counts_int)
write_map(OUT_FILE, f'star_counts/star_counts_diff_{NSIDE}', star_counts_diff)
# Store the distance edges alongside the maps
write_map(OUT_FILE, 'distance_bins', dist_edges)

## 7. Read the maps back

`HDF5Loader` reads arbitrary dataset paths and caches them, the same way
`StellarCatalogLoader` caches catalog columns.

In [ ]:
my_maps = loader.HDF5Loader(data_dir, OUT_FILE)
E_map_check = my_maps.load_data([f'E_map/E_map_int_{NSIDE}'])
print(E_map_check.shape, np.allclose(E_map_check, E_map_int, equal_nan=True))

In [ ]:
# What ended up in the file?
with h5.File(os.path.join(data_dir, OUT_FILE), 'r') as f:
    f.visititems(lambda name, obj: print(f"{name}  {obj.shape}")
                 if isinstance(obj, h5.Dataset) else None)

## 8. Plot the maps

Below is a copy of the plotting function in `Zhang_Green_maps.ipynb`.

Be careful to pass the appropriate values for the coordinate ('G' or 'C') and nest in the call to orthview.

In [ ]:
def plot_int_ext_full(ext_map, title, nest=True, cmb_contour=False, gal_contour=False,
                      vmin=None, vmax=None, cbar_label="$E$ [mag]",
                      save=False, fname=None):
    """Plot a full-sky map in galactic coordinates."""
    hp.orthview(ext_map,
                coord=['G'],
                rot=[0, 0, 0],
                nest=nest,
                title=title,
                cmap='inferno',
                badcolor='white',
                min=vmin, max=vmax,
                cbar=False,
                notext=True,
                hold=True)
    # Graticule
    hp.graticule(dpar=10, dmer=20)
    # Axes
    ax = plt.gca()
    ax.text(0.0, -1 - 0.06, "Galactic longitude [deg]", fontsize=10,
            va='top', ha='center', color='black')
    ax.text(-2 - 0.2, 0.0, "Galactic latitude [deg]", fontsize=10, rotation=90,
            va='center', ha='left', color='black')
    # Colorbar
    im = ax.get_images()[0]
    cbar = plt.colorbar(im, ax=ax, shrink=0.5)
    cbar.set_label(cbar_label, fontsize=10, labelpad=10)
    # Contours
    for draw, field in ((cmb_contour, cmb), (gal_contour, gal)):
        if not draw:
            continue
        xgrid, ygrid = field.get_mesh(galactic=True)
        proj = ax.proj
        x_contour, y_contour = proj.ang2xy(xgrid.ravel(), ygrid.ravel(), lonlat=True)
        x_contour = x_contour.reshape(xgrid.shape)
        y_contour = y_contour.reshape(ygrid.shape)
        ax.contour(x_contour, y_contour, field.Pw, levels = [0.001], colors = 'white')
    # Save or show
    if save:
        plt.savefig(os.path.join(plt_dir, fname), bbox_inches='tight')
        plt.close()
    else:
        plt.show()

In [ ]:
bin_num = 12
plt_title = f"Extinction at bin {bin_num} ({all_distances[bin_num]:.2f} kpc - {all_distances[bin_num + 1]:.2f} kpc)"
plot_int_ext_full(E_map_int[bin_num],
                  plt_title,
                  nest=NEST, vmin=0, vmax=1.6, cmb_contour=True, gal_contour=True)

In [ ]:
plot_int_ext_full(star_counts_int, "Total star counts", nest=NEST,
                  vmin=0, vmax=np.nanpercentile(star_counts_int, 99),
                  cbar_label="counts")